# Fit the CARLA behavior-v2 controller
Use a CPU runtime. Build the upload ZIP in the checkout with `python tools/reproduce_behavior_v2.py --colab-zip out/behavior-v2-colab.zip`. The fit uses 17,236 labeled pairs and verifies the frozen model hash before download.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile
from io import BytesIO
from google.colab import files

uploads = files.upload()
if len(uploads) != 1:
    raise ValueError("Upload one behavior-v2 input ZIP")
project = Path(mkdtemp(prefix="ado-behavior-v2-"))
with ZipFile(BytesIO(next(iter(uploads.values())))) as archive:
    archive.extractall(project)
print(project)

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "data/behavior-v2/requirements-colab.txt"],
               cwd=project, check=True)
subprocess.run([sys.executable, "tools/reproduce_behavior_v2.py"], cwd=project, check=True)

In [ ]:
from zipfile import ZIP_DEFLATED

outputs = project / "out/behavior-v2-refit"
download = project / "behavior-v2-verified.zip"
with ZipFile(download, "w", ZIP_DEFLATED) as archive:
    for filename in ("candidate.npz", "training-report.json"):
        archive.write(outputs / filename, filename)
files.download(str(download))